# 06  —  SQL Analysis with DuckDB

Demonstrate analytical SQL on the VM fleet data using DuckDB.
DuckDB reads CSVs directly — no database server needed.

Skills shown: CTEs, window functions, aggregations, JOINs, CASE expressions, percentiles.

In [1]:
import duckdb
import pandas as pd
import plotly.express as px

con = duckdb.connect()

con.execute("""
    CREATE VIEW summary AS SELECT * FROM read_csv_auto('../data/clean/vm_utilization_summary.csv');
    CREATE VIEW classified AS SELECT * FROM read_csv_auto('../data/clean/vm_classified.csv');
    CREATE VIEW recommendations AS SELECT * FROM read_csv_auto('../data/clean/vm_recommendations.csv');
    CREATE VIEW fleet_cost AS SELECT * FROM read_csv_auto('../data/clean/fleet_cost_estimate.csv');
""")
print('Views created.')
con.execute('SELECT COUNT(*) AS total_vms FROM summary').df()

Views created.


,total_vms
0,123363


## 1. Fleet overview — waste by VM class

In [2]:
con.execute("""
    SELECT
        class,
        COUNT(*)                          AS vm_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
        ROUND(AVG(cpu_mean), 2)           AS avg_cpu,
        ROUND(AVG(mem_mean), 2)           AS avg_mem
    FROM classified
    GROUP BY class
    ORDER BY vm_count DESC
""").df()

,class,vm_count,pct,avg_cpu,avg_mem
0,idle,70434,57.1,4.11,72.45
1,oversized,33470,27.1,12.42,93.10
2,zombie,13499,10.9,1.47,10.83
3,right-sized,5456,4.4,42.43,83.76
4,review,288,0.2,15.28,89.89
5,hot,216,0.2,85.21,61.37


## 2. Top 20 most wasteful VMs (highest savings potential)

Window function ranks VMs within each action category.

In [3]:
con.execute("""
    WITH ranked AS (
        SELECT
            instance,
            action,
            risk,
            monthly_savings,
            ROW_NUMBER() OVER (
                PARTITION BY action
                ORDER BY monthly_savings DESC
            ) AS rank_in_action
        FROM recommendations
        WHERE monthly_savings > 0
    )
    SELECT *
    FROM ranked
    WHERE rank_in_action <= 5
    ORDER BY monthly_savings DESC
""").df()

,instance,action,risk,monthly_savings,rank_in_action
0,instance-4dc8f0c6-4d3a-4e62-a4c0-b1567bc787f5,downsize,moderate,50.798860,1
1,instance-f4ed10a9-4d03-49e1-882d-bcde37f60db7,downsize,moderate,48.923207,2
2,instance-af789b6d-f894-40d9-84c3-225a03036f0c,downsize,moderate,47.642185,3
3,instance-6ea6710f-91ce-4098-bd79-59a593f9e18d,review,n/a,44.671766,1
4,instance-0c426b65-473d-4a61-bc17-b781477d68ba,terminate,moderate,40.088458,1
5,instance-f820bfdd-f470-4379-a15a-23ad3c47ec60,review,n/a,34.037796,2
6,instance-2b4ac037-6aa2-4556-a79e-62d98f08598e,review,n/a,32.566061,3
7,instance-25e2878e-70e7-40f5-8449-041a36fcc4fc,review,n/a,31.120137,4
8,instance-2c2b9925-95a4-4029-845d-7dd968776bc8,review,n/a,28.908196,5
9,instance-256da4d3-908c-40d5-b288-e45c7d6667e1,downsize,moderate,28.753585,4


## 3. CPU utilization distribution — percentile buckets

NTILE window function to split VMs into deciles by CPU usage.

In [4]:
df_deciles = con.execute("""
    WITH deciles AS (
        SELECT
            instance,
            cpu_mean,
            NTILE(10) OVER (ORDER BY cpu_mean) AS decile
        FROM summary
    )
    SELECT
        decile,
        COUNT(*)                    AS n_vms,
        ROUND(MIN(cpu_mean), 2)     AS cpu_min,
        ROUND(MAX(cpu_mean), 2)     AS cpu_max,
        ROUND(AVG(cpu_mean), 2)     AS cpu_avg
    FROM deciles
    GROUP BY decile
    ORDER BY decile
""").df()
print(df_deciles.to_string(index=False))

fig = px.bar(df_deciles, x='decile', y='n_vms', text='cpu_avg',
             labels={'decile': 'CPU Decile', 'n_vms': 'VM Count'},
             title='VM count per CPU utilization decile (avg CPU shown)')
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.show()

 decile  n_vms  cpu_min  cpu_max  cpu_avg
      1  12337     0.00     1.20     0.65
      2  12337     1.20     2.17     1.71
      3  12337     2.17     2.89     2.52
      4  12336     2.89     3.72     3.28
      5  12336     3.72     4.95     4.31
      6  12336     4.95     6.52     5.68
      7  12336     6.52     8.52     7.51
      8  12336     8.52    12.07    10.03
      9  12336    12.07    15.92    14.20
     10  12336    15.92   100.00    29.49


## 4. JOIN — savings potential by VM class

Join classified VMs with recommendations to see savings breakdown per class.

In [5]:
con.execute("""
    SELECT
        c.class,
        COUNT(*)                              AS n_vms,
        SUM(r.monthly_savings)                AS total_savings,
        ROUND(AVG(r.monthly_savings), 2)      AS avg_savings,
        ROUND(MEDIAN(r.monthly_savings), 2)   AS median_savings,
        SUM(CASE WHEN r.risk = 'safe' THEN 1 ELSE 0 END) AS safe_count
    FROM classified c
    JOIN recommendations r ON c.instance = r.instance
    GROUP BY c.class
    ORDER BY total_savings DESC
""").df()

,class,n_vms,total_savings,avg_savings,median_savings,safe_count
0,oversized,17663,15178.834625,0.86,0.43,3051.0
1,right-sized,1586,4286.968022,2.70,0.81,1.0
2,idle,27149,3554.635700,0.13,0.00,14702.0
3,hot,28,507.779336,18.13,18.81,0.0
4,review,150,409.581059,2.73,0.00,11.0
5,zombie,477,15.492737,0.03,0.00,377.0


## 5. CTE pipeline — idle VMs with high memory (hidden waste)

Multi-step CTE: find VMs that look idle by CPU but use significant memory,
meaning they might be doing background work (caching, queues).

In [6]:
con.execute("""
    WITH cpu_idle AS (
        SELECT instance, cpu_mean, cpu_p95, mem_mean
        FROM summary
        WHERE cpu_mean < 5
    ),
    memory_active AS (
        SELECT *
        FROM cpu_idle
        WHERE mem_mean > 70
    ),
    with_class AS (
        SELECT
            m.*,
            c.class
        FROM memory_active m
        JOIN classified c ON m.instance = c.instance
    )
    SELECT
        class,
        COUNT(*)                      AS n_vms,
        ROUND(AVG(cpu_mean), 2)       AS avg_cpu,
        ROUND(AVG(mem_mean), 2)       AS avg_mem,
        ROUND(AVG(cpu_p95), 2)        AS avg_cpu_p95
    FROM with_class
    GROUP BY class
    ORDER BY n_vms DESC
""").df()

,class,n_vms,avg_cpu,avg_mem,avg_cpu_p95
0,idle,28912,2.78,91.74,3.88
1,oversized,990,4.09,91.78,13.10


## 6. Running totals — cumulative savings by action

Window function with ROWS BETWEEN for a running total of savings if VMs are addressed in order of savings potential.

In [7]:
df_cumul = con.execute("""
    WITH ordered AS (
        SELECT
            instance,
            action,
            monthly_savings,
            ROW_NUMBER() OVER (ORDER BY monthly_savings DESC) AS priority
        FROM recommendations
        WHERE monthly_savings > 0
    )
    SELECT
        priority,
        action,
        monthly_savings,
        SUM(monthly_savings) OVER (
            ORDER BY priority
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_savings
    FROM ordered
    WHERE priority <= 100
    ORDER BY priority
""").df()
print(f"Top 100 VMs account for ${df_cumul['cumulative_savings'].iloc[-1]:,.0f}/month in savings")

fig = px.line(df_cumul, x='priority', y='cumulative_savings', color='action',
              labels={'priority': 'VM priority rank', 'cumulative_savings': 'Cumulative savings ($)'},
              title='Cumulative savings: addressing VMs in order of impact')
fig.show()

Top 100 VMs account for $2,300/month in savings


## 7. CASE + GROUP BY — risk matrix

Cross-tabulate action and risk level to build an actionability matrix.

In [8]:
con.execute("""
    SELECT
        action,
        SUM(CASE WHEN risk = 'safe' THEN 1 ELSE 0 END)     AS safe,
        SUM(CASE WHEN risk = 'moderate' THEN 1 ELSE 0 END) AS moderate,
        SUM(CASE WHEN risk = 'risky' THEN 1 ELSE 0 END)    AS risky,
        COUNT(*) AS total,
        ROUND(SUM(monthly_savings), 0)                      AS total_savings
    FROM recommendations
    GROUP BY action
    ORDER BY total_savings DESC
""").df()

,action,safe,moderate,risky,total,total_savings
0,downsize,14021.0,6848.0,6930.0,27799,17040.0
1,review,0.0,0.0,0.0,2753,4687.0
2,terminate,4121.0,9133.0,3070.0,16324,1221.0
3,keep,0.0,0.0,0.0,177,1005.0


## 8. Advanced — prediction accuracy buckets

CASE expression to bin prediction error, then aggregate.

In [9]:
con.execute("""
    SELECT
        CASE
            WHEN ABS(actual_cpu - pred_mid) < 1  THEN '< 1%'
            WHEN ABS(actual_cpu - pred_mid) < 5  THEN '1-5%'
            WHEN ABS(actual_cpu - pred_mid) < 10 THEN '5-10%'
            ELSE '> 10%'
        END AS error_bucket,
        COUNT(*) AS n_vms,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
        ROUND(AVG(monthly_savings), 2) AS avg_savings
    FROM recommendations
    GROUP BY error_bucket
    ORDER BY
        CASE error_bucket
            WHEN '< 1%' THEN 1
            WHEN '1-5%' THEN 2
            WHEN '5-10%' THEN 3
            ELSE 4
        END
""").df()

,error_bucket,n_vms,pct,avg_savings
0,< 1%,27392,58.2,0.12
1,1-5%,15785,33.5,0.79
2,5-10%,3042,6.5,1.21
3,> 10%,834,1.8,5.46


## 9. Fleet cost by VM size — from cost estimate

Simple aggregation on the fleet cost table to show where money goes.

In [10]:
df_cost = con.execute("""
    SELECT
        size,
        ec2_type,
        vm_count,
        monthly_cost,
        zombie_savings + downsize_savings AS total_waste,
        ROUND(100.0 * (zombie_savings + downsize_savings) / monthly_cost, 1) AS waste_pct
    FROM fleet_cost
    WHERE monthly_cost > 0
    ORDER BY total_waste DESC
""").df()
print(df_cost.to_string(index=False))

fig = px.bar(df_cost, x='size', y=['monthly_cost', 'total_waste'],
             barmode='group',
             labels={'value': 'USD/month', 'size': 'VM Size'},
             title='Monthly cost vs recoverable waste by VM size')
fig.show()

                   size   ec2_type  vm_count  monthly_cost  total_waste  waste_pct
Extra Large/Extra Large r5.8xlarge      2009       2956605      1565131       52.9
          Medium/Medium   m5.large     37787       2648113      1552066       58.6
      Extra Large/Large r5.4xlarge      3476       2557780      1354682       53.0
           Medium/Small  t3.medium     74672       2267639      1201829       53.0
            Large/Large  r5.xlarge      1380        253865       134383       52.9
     Extra Large/Medium r5.2xlarge       455        167404        88485       52.9
            Small/Small   t3.small      2696         40936        21683       53.0
           Large/Medium   r5.large       760         69905        21536       30.8
           Medium/Large  m5.xlarge       125         17520         9180       52.4


## Summary

This notebook demonstrates analytical SQL on the cloud infrastructure dataset:
- **Aggregations + CASE**: fleet breakdown, risk matrix, error buckets
- **Window functions**: ROW_NUMBER ranking, NTILE deciles, running totals
- **CTEs**: multi-step filtering pipeline for hidden waste
- **JOINs**: enriching recommendations with VM classification

All queries run in-process via DuckDB — no database server required.